# 00 — Setup and demonstration data

Run this notebook **first**. It checks that the kernel has the scientific
stack, finds the `lowflow` package, and (only if `data/` is empty) writes the
synthetic demonstration workbooks.

**If you got `ModuleNotFoundError: No module named 'numpy'`:** the script ran
under an interpreter without the stack installed. Cell 1 prints the exact
install command for the kernel you are actually running.

In [ ]:
# --- 1. Which interpreter is this, and does it have the stack? -------------
# A "ModuleNotFoundError: No module named 'numpy'" means this kernel is not the
# environment you installed into. The pip line printed below targets THIS
# kernel, which is what you want -- a bare `pip install` may hit a different one.
import importlib.util, sys

print("interpreter :", sys.executable)
print("version     :", sys.version.split()[0])

REQUIRED = ["numpy", "scipy", "pandas", "matplotlib", "statsmodels",
            "openpyxl", "lmoments3"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

if missing:
    print("\nMISSING:", ", ".join(missing))
    print("\nInstall into THIS kernel by running the following in a terminal")
    print("(the quotes matter -- the path may contain spaces):\n")
    print(f'  "{sys.executable}" -m pip install ' + " ".join(missing))
    print("\nOr from inside this notebook, in a new cell:\n")
    print("  %pip install " + " ".join(missing))
    print("\nIf you use conda, prefer:")
    print("  conda install -n <env> " + " ".join(m for m in missing if m != "lmoments3"))
    print("  conda run -n <env> pip install lmoments3")
else:
    print("\nAll dependencies present.")

In [ ]:
# --- 2. Locate the lowflow package ----------------------------------------
# Searches this directory and its parents for lowflow/__init__.py, so the
# notebook works whether it sits beside the package or in a notebooks/
# subfolder. Set PKG_ROOT by hand if your layout is unusual.
import os, sys, pathlib

PKG_ROOT = None          # e.g. pathlib.Path("/path/to/lowflow-frequency-analysis")

if PKG_ROOT is None:
    here = pathlib.Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "lowflow" / "__init__.py").exists():
            PKG_ROOT = cand
            break

if PKG_ROOT is None:
    raise FileNotFoundError(
        "Could not find lowflow/__init__.py in this directory or any parent.\n"
        "Set PKG_ROOT above to the folder that CONTAINS the lowflow/ directory, "
        "e.g. pathlib.Path('/path/to/lowflow-frequency-analysis')"
    )

PKG_ROOT = pathlib.Path(PKG_ROOT).resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
os.chdir(PKG_ROOT)       # so relative data/ and out/ paths resolve

import lowflow
print("lowflow package :", lowflow.__file__)
print("working dir     :", os.getcwd())
print("version         :", lowflow.__version__)

## Demonstration data

The pipeline expects a daily series with columns `date` and `flow`, as
`.xlsx` or `.csv`. The cell below writes three synthetic
workbooks **only if they are not already there**, so it will never overwrite
your real records.

Replace `data/station_a.xlsx`, `data/station_b.xlsx` and `data/projection.xlsx` with
your own files when you are ready.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(PKG_ROOT / "scripts"))
data = pathlib.Path("data")
existing = sorted(p.name for p in data.glob("*.xlsx")) if data.exists() else []

if existing:
    print("data/ already contains:", ", ".join(existing))
    print("Leaving it untouched. Delete the files first if you want to regenerate.")
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "mk", PKG_ROOT / "scripts" / "00_make_synthetic_data.py")
    mk = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mk)
    mk.main("data")

In [ ]:
# quick look at whatever is in data/ now
rec = read_flow("data/station_a.xlsx", name="Station A", D=7) if False else None
import pandas as pd
for f in sorted(pathlib.Path("data").glob("*.xlsx")):
    df = pd.read_excel(f)
    print(f"{f.name:18s} {len(df):6d} rows  "
          f"{str(df['date'].min())[:10]} to {str(df['date'].max())[:10]}  "
          f"missing flow: {int(df['flow'].isna().sum())}")

Next: **01 — annual marginal analysis**.